# BeyondBench: Model Comparison

This notebook demonstrates how to evaluate multiple models and compare their performance.

**What you'll learn:**
- How to run evaluations for multiple models
- How to aggregate results for comparison
- How to visualize performance with matplotlib and pandas

**Prerequisites:**
```bash
pip install beyondbench[openai] matplotlib pandas
```

In [ ]:
import os
import json
import subprocess
from pathlib import Path

# Check for dependencies
try:
    import matplotlib.pyplot as plt
    import pandas as pd
    import numpy as np
    print("All visualization dependencies available")
except ImportError as e:
    print(f"Missing dependency: {e}")
    print("Install with: pip install matplotlib pandas numpy")

## 1. Simulate Multi-Model Results

For demonstration, we create synthetic results for multiple models.
Replace this with real evaluation results from `beyondbench evaluate`.

In [ ]:
import numpy as np

# Simulated accuracy results for 3 models across task suites
# In practice, run beyondbench evaluate for each model and load final_results.json
np.random.seed(42)

models = {
    "GPT-4o": {
        "easy": {"sum": 1.0, "mean": 0.98, "sorting": 0.97, "find_maximum": 1.0,
                 "median": 0.95, "mode": 0.92, "comparison": 1.0, "count_unique": 0.96},
        "medium": {"fibonacci_sequence": 0.90, "geometric_sequence": 0.88,
                   "matrix_operations": 0.85, "polynomial_evaluation": 0.82,
                   "combinatorics": 0.87},
        "hard": {"knapsack": 0.72, "sudoku_solving": 0.68,
                 "shortest_path": 0.75, "edit_distance": 0.80,
                 "coin_change": 0.78},
    },
    "Llama-3.2-3B": {
        "easy": {"sum": 0.88, "mean": 0.82, "sorting": 0.75, "find_maximum": 0.91,
                 "median": 0.78, "mode": 0.70, "comparison": 0.95, "count_unique": 0.73},
        "medium": {"fibonacci_sequence": 0.62, "geometric_sequence": 0.55,
                   "matrix_operations": 0.48, "polynomial_evaluation": 0.51,
                   "combinatorics": 0.58},
        "hard": {"knapsack": 0.32, "sudoku_solving": 0.28,
                 "shortest_path": 0.38, "edit_distance": 0.42,
                 "coin_change": 0.35},
    },
    "Claude-Sonnet-4": {
        "easy": {"sum": 0.99, "mean": 0.97, "sorting": 0.96, "find_maximum": 0.99,
                 "median": 0.94, "mode": 0.91, "comparison": 1.0, "count_unique": 0.95},
        "medium": {"fibonacci_sequence": 0.92, "geometric_sequence": 0.90,
                   "matrix_operations": 0.87, "polynomial_evaluation": 0.85,
                   "combinatorics": 0.89},
        "hard": {"knapsack": 0.76, "sudoku_solving": 0.72,
                 "shortest_path": 0.78, "edit_distance": 0.83,
                 "coin_change": 0.80},
    },
}

print(f"Models: {list(models.keys())}")
print(f"Suites: {list(list(models.values())[0].keys())}")

## 2. Running Real Evaluations (Optional)

If you have API keys, run real evaluations here:

In [ ]:
# Example: run two models in sequence via CLI
# Uncomment and set your API keys first

# models_to_eval = [
#     ("gpt-4o",         "openai",    os.environ.get("OPENAI_API_KEY", "")),
#     ("claude-sonnet-4-20250514", "anthropic", os.environ.get("ANTHROPIC_API_KEY", "")),
# ]

# results_dirs = {}
# for model_id, provider, key in models_to_eval:
#     if not key:
#         print(f"Skipping {model_id}: no API key")
#         continue
#     out_dir = f"/tmp/bb_compare/{model_id.replace('/', '_')}"
#     cmd = [
#         "beyondbench", "evaluate",
#         "--model-id", model_id,
#         "--api-provider", provider,
#         "--api-key", key,
#         "--suite", "easy",
#         "--datapoints", "20",
#         "--output-dir", out_dir,
#     ]
#     subprocess.run(cmd, check=True)
#     results_dirs[model_id] = out_dir
#     print(f"Completed {model_id}")

print("Using simulated results for this demo")

## 3. Build a Comparison DataFrame

In [ ]:
import pandas as pd

# Build a flat DataFrame: model x task -> accuracy
rows = []
for model_name, suites in models.items():
    for suite_name, tasks in suites.items():
        for task_name, accuracy in tasks.items():
            rows.append({
                "model": model_name,
                "suite": suite_name,
                "task": task_name,
                "accuracy": accuracy,
            })

df = pd.DataFrame(rows)
print(f"DataFrame shape: {df.shape}")
df.head(10)

In [ ]:
# Suite-level summary
suite_summary = df.groupby(["model", "suite"])["accuracy"].mean().unstack()
print("Average accuracy per model and suite:")
print(suite_summary.round(3).to_string())

## 4. Bar Chart: Suite-Level Comparison

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

fig, ax = plt.subplots(figsize=(10, 6))

model_names = list(models.keys())
suites = ["easy", "medium", "hard"]
suite_colors = {"easy": "#2ecc71", "medium": "#f39c12", "hard": "#e74c3c"}

x = np.arange(len(model_names))
width = 0.25

for i, suite in enumerate(suites):
    values = [suite_summary.loc[m, suite] if m in suite_summary.index else 0
              for m in model_names]
    bars = ax.bar(x + i * width, values, width, label=suite.capitalize(),
                  color=suite_colors[suite], alpha=0.85, edgecolor="white")

    # Add value labels
    for bar, val in zip(bars, values):
        ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.005,
                f"{val:.0%}", ha="center", va="bottom", fontsize=9)

ax.set_xticks(x + width)
ax.set_xticklabels(model_names, fontsize=11)
ax.set_ylim(0, 1.12)
ax.set_ylabel("Average Accuracy", fontsize=12)
ax.set_title("BeyondBench: Model Comparison by Suite", fontsize=14, fontweight="bold")
ax.legend(title="Suite", fontsize=10)
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda y, _: f"{y:.0%}"))
ax.grid(axis="y", alpha=0.3)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)

plt.tight_layout()
plt.savefig("/tmp/model_comparison_suite.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved to /tmp/model_comparison_suite.png")

## 5. Heatmap: Per-Task Accuracy

In [ ]:
# Pivot for heatmap: rows = tasks, columns = models
pivot = df.pivot_table(index="task", columns="model", values="accuracy")
pivot = pivot.sort_values(pivot.columns[0], ascending=False)

fig, ax = plt.subplots(figsize=(10, 10))

im = ax.imshow(pivot.values, cmap="RdYlGn", vmin=0, vmax=1, aspect="auto")

ax.set_xticks(range(len(pivot.columns)))
ax.set_xticklabels(pivot.columns, fontsize=11, rotation=15)
ax.set_yticks(range(len(pivot.index)))
ax.set_yticklabels(pivot.index, fontsize=9)
ax.set_title("BeyondBench: Accuracy Heatmap (per task)", fontsize=13, fontweight="bold", pad=15)

# Annotate cells
for i in range(len(pivot.index)):
    for j in range(len(pivot.columns)):
        val = pivot.values[i, j]
        text_color = "white" if val < 0.5 else "black"
        ax.text(j, i, f"{val:.0%}", ha="center", va="center",
                fontsize=9, color=text_color, fontweight="bold")

plt.colorbar(im, ax=ax, shrink=0.8, label="Accuracy")
plt.tight_layout()
plt.savefig("/tmp/model_comparison_heatmap.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved to /tmp/model_comparison_heatmap.png")

## 6. Radar Chart: Overall Capability Profile

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# One data point per suite
categories = ["Easy", "Medium", "Hard"]
N = len(categories)
angles = np.linspace(0, 2 * np.pi, N, endpoint=False).tolist()
angles += angles[:1]  # close the polygon

model_colors = {
    "GPT-4o": "#3498db",
    "Llama-3.2-3B": "#e74c3c",
    "Claude-Sonnet-4": "#2ecc71",
}

fig, ax = plt.subplots(figsize=(7, 7), subplot_kw=dict(polar=True))

for model_name, suites in models.items():
    values = [
        np.mean(list(suites["easy"].values())),
        np.mean(list(suites["medium"].values())),
        np.mean(list(suites["hard"].values())),
    ]
    values += values[:1]
    color = model_colors.get(model_name, "gray")
    ax.plot(angles, values, "o-", linewidth=2, label=model_name, color=color)
    ax.fill(angles, values, alpha=0.1, color=color)

ax.set_xticks(angles[:-1])
ax.set_xticklabels(categories, fontsize=13)
ax.set_ylim(0, 1)
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda y, _: f"{y:.0%}"))
ax.set_title("Model Capability Radar", fontsize=14, fontweight="bold", pad=20)
ax.legend(loc="upper right", bbox_to_anchor=(1.35, 1.1))
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig("/tmp/model_radar.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved to /tmp/model_radar.png")

## 7. Comparative Table

In [ ]:
import pandas as pd

# Build comparison table
comparison_data = []
for model_name, suites in models.items():
    row = {"Model": model_name}
    for suite_name, tasks in suites.items():
        row[f"{suite_name.capitalize()} Avg"] = f"{np.mean(list(tasks.values())):.1%}"
    all_acc = [v for s in suites.values() for v in s.values()]
    row["Overall"] = f"{np.mean(all_acc):.1%}"
    comparison_data.append(row)

comparison_df = pd.DataFrame(comparison_data)
comparison_df = comparison_df.set_index("Model")

print("Model Comparison Summary:")
print(comparison_df.to_string())

## 8. Loading Real Results from JSON

If you ran real evaluations, load them like this:

In [ ]:
def load_beyondbench_results(results_dir: str) -> dict:
    """
    Load final_results.json from a BeyondBench output directory.
    Returns a dict: {task_name: accuracy}
    """
    results_file = Path(results_dir) / "final_results.json"
    if not results_file.exists():
        raise FileNotFoundError(f"No results file at {results_file}")

    with open(results_file) as f:
        raw = json.load(f)

    task_accuracies = {}
    for task_name, task_data in raw.get("task_results", {}).items():
        if isinstance(task_data, list) and task_data:
            acc = sum(m.get("accuracy", 0) for m in task_data) / len(task_data)
        elif isinstance(task_data, dict):
            summary = task_data.get("summary", {})
            acc = summary.get("avg_accuracy", task_data.get("overall_accuracy", 0))
        else:
            acc = 0.0
        task_accuracies[task_name] = acc

    return task_accuracies


# Usage (after running real evaluations):
# results = {
#     "gpt-4o":            load_beyondbench_results("/tmp/bb_compare/gpt-4o"),
#     "claude-sonnet-4":   load_beyondbench_results("/tmp/bb_compare/claude-sonnet-4-20250514"),
# }

print("load_beyondbench_results() defined — use it to load your real evaluation outputs")

## Summary

You've seen how to:
- Evaluate multiple models using the CLI or Python API
- Build a comparison DataFrame from results
- Visualize with bar charts, heatmaps, and radar charts
- Use `load_beyondbench_results()` to parse real JSON output

**Next:** `04_result_analysis.ipynb` — Deep statistical analysis of evaluation results.